# Notebook 6: External Validation

**Purpose:** Genuine external predictivity, evaluated across all 5 targets, on
two independent axes rather than one:

1. **BindingDB** -- a different bioactivity database entirely, not just a
   held-out split of the same ChEMBL pull. Uses a UniProt-ID-keyed fetch with a SHA-256 lock-file mechanism (fetched once,
   hash-verified on every subsequent run so the validation set can never
   silently drift).
2. **Temporal holdout** -- ChEMBL records published after a per-target cutoff
   year, evaluated with the *already-deployed* model (no retraining). This is
   the harder, more realistic claim (would this model have predicted well on
   compounds that didn't exist yet when it was trained?), but comes with a
   real caveat: notebook 3's deployed models used **scaffold** splits, not
   temporal ones, so some post-cutoff compounds may already have been inside
   training. That contamination is checked and disclosed explicitly per
   compound, not assumed away -- results are reported separately for
   genuinely-never-seen vs. was-in-training-anyway compounds. A fully
   temporally-blind retrained model would be the gold-standard version of
   this analysis; this is the tractable first version, scoped like notebook
   3B was (bounded, disclosed, not silently presented as more than it is).

**Why this notebook matters, not just "more validation":** the OECD's five
QSAR model validation principles (Tropsha 2010, Gramatica 2007) are the
standard framework reviewers measure a QSAR paper against -- defined
endpoint, unambiguous algorithm, defined applicability domain, goodness-of-
fit/robustness/predictivity, and mechanistic interpretation. This project's
notebooks 2-5 already satisfy the first four internally; genuine external
validation is what closes principle 4 for real, and only ~10% of published
predictive-modelling studies include one at all.

**Explicitly reuses, does not reinvent:** notebook 3's real feature recipe
(Morgan r=2/2048 bits + 10 physicochemical descriptors + combined), notebook
3's real deployed models/Venn-ABERS calibrators/conformal wrappers/AD
references (not an incompatible earlier 
calibration approach), and notebook 2's real structure-standardisation
function (so an external compound is cleaned identically to how training
compounds were, not by a second, silently-different implementation).

**Deliberately out of scope for this first pass:** a fully retrained
temporally-blind model (flagged above as the gold-standard follow-on, not
a blocker); DrugBank appears nowhere here (that's notebook 4's job, a
prospective screening question, not a retrospective validation one).

In [1]:
# MUST BE FIRST CELL!
import os
import multiprocessing

HPC_MODE = False

if HPC_MODE:
    N_CORES = int(os.environ.get("NCPUS") or os.environ.get("PBS_NP") or
                  os.environ.get("PBS_NCPUS") or os.environ.get("SLURM_CPUS_PER_TASK") or
                  multiprocessing.cpu_count())
    import matplotlib
    matplotlib.use("Agg")
else:
    N_CORES = min(multiprocessing.cpu_count(), 4)

for var in ["OMP_NUM_THREADS", "MKL_NUM_THREADS", "OPENBLAS_NUM_THREADS",
            "NUMEXPR_NUM_THREADS", "VECLIB_MAXIMUM_THREADS"]:
    os.environ[var] = "1" if HPC_MODE else str(N_CORES)
os.environ["OMP_NESTED"] = "FALSE"
os.environ["MKL_DYNAMIC"] = "FALSE"

ENV = "HPC" if HPC_MODE else "Colab"
print(f"Environment: {ENV} | CV workers: {N_CORES}")

Environment: Colab | CV workers: 2


In [14]:
from pathlib import Path

if HPC_MODE:
    PROJECT_DIR = Path("./")
else:
    from google.colab import drive
    drive.mount("/content/drive")
    PROJECT_DIR = Path("/content/drive/My Drive/gpcr_benchmark")

for subdir in ["data/processed", "data/external/bindingdb", "ml/results/external_validation"]:
    (PROJECT_DIR / subdir).mkdir(parents=True, exist_ok=True)

data_path = PROJECT_DIR / "data"
results_path = PROJECT_DIR / "ml" / "results"
models_path = PROJECT_DIR / "ml" / "models"
ext_path = results_path / "external_validation"
ext_data_path = data_path / "external" / "bindingdb"

print(f"PROJECT_DIR: {PROJECT_DIR}")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
PROJECT_DIR: /content/drive/My Drive/gpcr_benchmark


In [3]:
# Not preinstalled on a fresh Colab runtime (xgboost/lightgbm/sklearn/joblib/
# pandas/numpy are). Safe to rerun -- pip no-ops if already satisfied.
if not HPC_MODE:
    import subprocess
    subprocess.run(['pip', 'install', '-q', 'rdkit', 'venn_abers', 'crepes'], check=True)
    print('Install check done.')

Install check done.


In [15]:
import json
import hashlib
import datetime
import platform
import subprocess
import urllib.request
import urllib.error
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd

from rdkit import Chem, RDLogger
from rdkit.Chem import AllChem, Descriptors, Crippen, Lipinski
from rdkit.Chem.MolStandardize import rdMolStandardize
RDLogger.DisableLog("rdApp.*")

import joblib
import venn_abers

# sklearn re-emits this on every unpickled estimator regardless of the blanket
# filter above (each of RandomForest's ~100+ internal DecisionTreeClassifier
# sub-estimators triggers one) -- these artifacts are pickled under a
# slightly older sklearn than the Colab runtime ships; harmless for
# tree-based/linear estimators used here, but silence the resulting log spam
# explicitly rather than relying on the general filter to catch it.
try:
    from sklearn.exceptions import InconsistentVersionWarning
    warnings.filterwarnings("ignore", category=InconsistentVersionWarning)
except ImportError:
    pass

print("Imports OK.")
print("Python:", platform.python_version())
for pkg in ("rdkit", "numpy", "pandas", "sklearn", "xgboost", "lightgbm", "venn_abers", "crepes", "joblib"):
    try:
        mod = __import__(pkg)
        print(f"{pkg}: {getattr(mod, '__version__', 'unknown')}")
    except ImportError:
        print(f"{pkg}: NOT INSTALLED")


Imports OK.
Python: 3.13.15
rdkit: 2026.03.5
numpy: 2.1.3
pandas: 2.2.3
sklearn: 1.6.1
xgboost: 3.4.1
lightgbm: 4.6.0
venn_abers: unknown
crepes: 0.9.1
joblib: 1.5.3


## Module A: Configuration -- 5 targets, matching notebook 3 exactly

Every constant below is copied verbatim from notebook 3's own config cell,
not re-typed from memory -- a mismatched Morgan radius or descriptor list
would silently produce a feature vector the deployed model was never
trained on. **`Kd` is deliberately excluded from the BindingDB fetch** (`BINDINGDB_AFFINITY_TYPES = ['Ki', 'IC50', 'EC50']` only) -- the deployed models were never trained on Kd measurements, so scoring against them would validate against an endpoint class this project doesn't actually predict.

In [19]:
RANDOM_STATE = 42
ACTIVITY_THRESHOLD = 6.0  # must match notebook 02/03 exactly

TARGETS = ['drd2', 'cb2', 'adora2a', 'oprm1', 'ccr5']
POOL = 'full'  # notebook 4's screening deployment pool -- matches what's actually served
REPRESENTATION = 'combined'  # notebook 3's PRIMARY_REPRESENTATION for deployment

MORGAN_RADIUS = 2
MORGAN_NBITS = 2048
DESCRIPTOR_COLS = ['MW', 'LogP', 'TPSA', 'HBD', 'HBA', 'RotBonds',
                   'HeavyAtomCount', 'RingCount', 'AromaticRingCount', 'FractionCSP3']

# UniProt IDs -- live-verified against UniProt directly (2026-08), not assumed.
TARGET_UNIPROT = {
    'drd2': 'P14416',     # D(2) dopamine receptor, human
    'cb2': 'P34972',      # Cannabinoid receptor 2, human
    'adora2a': 'P29274',  # Adenosine receptor A2a, human
    'oprm1': 'P35372',    # Mu-type opioid receptor, human
    'ccr5': 'P51681',     # C-C chemokine receptor type 5, human
}

# BindingDB affinity types pulled per target -- matches this project's own
# "full" pool convention (Ki + IC50 + EC50 pooled, not just Ki alone like
# the predecessor's single-type fetch).
BINDINGDB_AFFINITY_TYPES = ['Ki', 'IC50', 'EC50']
BINDINGDB_CUTOFF_NM = 1000000  # 1 mM -- broad BindingDB retrieval cap matching notebook
                                # 02's modeled-data quality window (0.1 nM - 1 mM).
                                # Notebook 01's ChEMBL pull itself had no upper-bound filter,
                                # but notebook 02 removed values outside this biologically
                                # plausible range before any ML modeling. Was 10,000 nM
                                # (10 uM) originally, a self-imposed restriction with no
                                # basis in the final training data scope. That 10 uM ceiling
                                # also skewed the BindingDB harmonised sets toward actives:
                                # weak/inactive compounds and genuinely-new-to-training
                                # compounds among them were silently excluded by construction.
                                # ACTIVITY_THRESHOLD (pActivity >= 6.0) remains the real
                                # active/inactive cut, applied after harmonisation as before.

# Temporal holdout: per-target cutoff year, chosen from each target's real
# document_year distribution (Module C), not a single arbitrary global year --
# disclosed per target once computed, not assumed identical across targets.
TEMPORAL_HOLDOUT_FRACTION_TARGET = 0.20  # aim: ~20% of each target's compounds
                                          # fall after the cutoff, adjusted per
                                          # target from the real distribution

print(f'Targets: {TARGETS}')
print(f'Pool: {POOL} | Representation: {REPRESENTATION}')
print(f'UniProt IDs: {TARGET_UNIPROT}')

Targets: ['drd2', 'cb2', 'adora2a', 'oprm1', 'ccr5']
Pool: full | Representation: combined
UniProt IDs: {'drd2': 'P14416', 'cb2': 'P34972', 'adora2a': 'P29274', 'oprm1': 'P35372', 'ccr5': 'P51681'}


## Module B: BindingDB Fetch + Lock (per target)

Uses the established mechanism: fetch once via
BindingDB's REST API, SHA-256-lock the raw file so it can never silently
change between runs, generalised here from a single hardcoded CB2 pull to a
loop over all 5 targets. The response-key misspelling
(`getLindsByUniprotResponse`, confirmed against a live query, not a typo
introduced here) and censored-value (`>`/`<` qualified) exclusion are both
carried over unchanged -- both are real properties of BindingDB's API, not
bugs in an earlier implementation's code.

In [20]:
def fetch_bindingdb_raw(target_key: str, uniprot_id: str) -> pd.DataFrame:
    '''ONE BindingDB query per target (not per affinity type -- the API
    response is not filterable by affinity type server-side, so looping the
    request itself only re-fetches identical data 3x for no reason; fetch
    once, filter locally for all of BINDINGDB_AFFINITY_TYPES in one pass.
    Kd is deliberately excluded: BINDINGDB_AFFINITY_TYPES = [Ki, IC50, EC50]
    only, matching exactly what the deployed models were trained on (notebook
    2/3's "full" pool convention) -- Kd records are real in BindingDB's
    response but are never requested, so they are silently absent from
    all_records by construction, not filtered out after the fact.'''
    url = (f'https://bindingdb.org/rest/getLigandsByUniprot'
           f'?uniprot={uniprot_id};{BINDINGDB_CUTOFF_NM}&response=application/json')
    try:
        with urllib.request.urlopen(url, timeout=60) as resp:
            bdb_data = json.load(resp)
    except (urllib.error.URLError, urllib.error.HTTPError) as exc:
        print(f'  \u26a0\ufe0f  {target_key}: fetch failed -- {exc}')
        return pd.DataFrame()

    response_key = next(iter(bdb_data.keys()))
    affinities = bdb_data[response_key].get('bdb.affinities', [])

    all_records = []
    n_wrong_type = n_censored = n_bad_value = 0
    for a in affinities:
        if a['bdb.affinity_type'] not in BINDINGDB_AFFINITY_TYPES:
            n_wrong_type += 1  # includes Kd and anything else not requested
            continue
        raw_val = a['bdb.affinity'].strip()
        if '>' in raw_val or '<' in raw_val:
            n_censored += 1
            continue
        try:
            val = float(raw_val)
        except ValueError:
            n_bad_value += 1
            continue
        if val <= 0:
            continue
        all_records.append({'SMILES': a['bdb.smile'], 'affinity_type': a['bdb.affinity_type'],
                             'affinity_nm': val, 'monomerid': a['bdb.monomerid']})

    print(f'  {target_key}: {len(affinities)} total records, '
          f'{n_wrong_type} not in {BINDINGDB_AFFINITY_TYPES} (incl. any Kd), '
          f'{n_censored} censored, {n_bad_value} unparseable, {len(all_records)} usable')
    return pd.DataFrame(all_records)


def _sha256_file(path):
    h = hashlib.sha256()
    with open(path, 'rb') as f:
        for chunk in iter(lambda: f.read(1 << 20), b''):
            h.update(chunk)
    return h.hexdigest()


bindingdb_raw = {}
bindingdb_cutoff_tag = f'cutoff_{int(BINDINGDB_CUTOFF_NM)}nM'

for target in TARGETS:
    # Include cutoff in the raw filename. This prevents a 1 mM run from silently
    # reusing older 10 uM-locked raw files, which would make the notebook text
    # and the actual validation set disagree.
    raw_path = ext_data_path / f'bindingdb_raw_{target}_{bindingdb_cutoff_tag}.csv'
    lock_path = ext_data_path / f'bindingdb_raw_{target}_{bindingdb_cutoff_tag}.lock.json'

    if raw_path.exists():
        print(f'{target}: {raw_path.name} already exists -- skipping fetch (using existing file).')
        df_bdb = pd.read_csv(raw_path)
    else:
        print(f'Fetching BindingDB for {target.upper()} (UniProt {TARGET_UNIPROT[target]})...')
        df_bdb = fetch_bindingdb_raw(target, TARGET_UNIPROT[target])
        if len(df_bdb) == 0:
            print(f'  \u26a0\ufe0f  {target}: zero usable BindingDB records -- flagged, not raised, '
                  f'so other targets are unaffected. This target will be excluded from the '
                  f'BindingDB axis (temporal axis is unaffected).')
        df_bdb.to_csv(raw_path, index=False)
        print(f'  Saved: {raw_path} ({len(df_bdb)} rows, '
              f'{df_bdb["SMILES"].nunique() if len(df_bdb) else 0} unique SMILES)')

    bindingdb_raw[target] = df_bdb

    # Lock check -- same "must not change after first run" convention as
    # every other locked artifact in this project.
    current_hash = _sha256_file(raw_path)
    if lock_path.exists():
        with open(lock_path) as f:
            lock = json.load(f)
        if lock['sha256'] != current_hash:
            print(f'  \u26a0\ufe0f\u26a0\ufe0f\u26a0\ufe0f  WARNING: {raw_path.name} changed since '
                  f'locked on {lock["locked_at"]} -- this violates the locked-dataset requirement.')
        else:
            print(f'  \u2705 {target}: BindingDB dataset unchanged since lock ({lock["locked_at"]}).')
    else:
        lock = {'sha256': current_hash, 'locked_at': datetime.datetime.now().isoformat(),
                'target': target, 'uniprot_id': TARGET_UNIPROT[target], 'path': str(raw_path),
                'bindingdb_cutoff_nm': BINDINGDB_CUTOFF_NM,
                'bindingdb_affinity_types': BINDINGDB_AFFINITY_TYPES}
        with open(lock_path, 'w') as f:
            json.dump(lock, f, indent=2)
        print(f'  \U0001f512 {target}: BindingDB dataset locked for the first time.')

print()
for target in TARGETS:
    print(f'{target}: {len(bindingdb_raw[target])} raw BindingDB records')

drd2: bindingdb_raw_drd2_cutoff_1000000nM.csv already exists -- skipping fetch (using existing file).
  ✅ drd2: BindingDB dataset unchanged since lock (2026-08-25T06:58:56.027038).
cb2: bindingdb_raw_cb2_cutoff_1000000nM.csv already exists -- skipping fetch (using existing file).
  ✅ cb2: BindingDB dataset unchanged since lock (2026-08-25T06:59:00.900824).
adora2a: bindingdb_raw_adora2a_cutoff_1000000nM.csv already exists -- skipping fetch (using existing file).
  ✅ adora2a: BindingDB dataset unchanged since lock (2026-08-25T06:59:05.778643).
oprm1: bindingdb_raw_oprm1_cutoff_1000000nM.csv already exists -- skipping fetch (using existing file).
  ✅ oprm1: BindingDB dataset unchanged since lock (2026-08-25T06:59:11.636722).
ccr5: bindingdb_raw_ccr5_cutoff_1000000nM.csv already exists -- skipping fetch (using existing file).
  ✅ ccr5: BindingDB dataset unchanged since lock (2026-08-25T06:59:12.174959).

drd2: 569 raw BindingDB records
cb2: 507 raw BindingDB records
adora2a: 524 raw Bindi

## Module C: Harmonisation -- BindingDB Axis

Standardises BindingDB SMILES through the **exact same chain** notebook 2
uses for training data (`Cleanup -> FragmentParent -> Normalizer ->
Uncharger -> TautomerEnumerator.Canonicalize`, copied verbatim, not
reimplemented) -- an external compound cleaned by a subtly different
pipeline would break the training-set overlap check (Module D) and could
silently shift the feature distribution the model sees. Multiple affinity
measurements for the same compound are pooled to a single potency value
the same way notebook 2 pools Ki/IC50/EC50 (median of -log10(M) values),
and the same `ACTIVITY_THRESHOLD` defines the binary label.

In [21]:
def standardize_smiles(smiles: str):
    '''Identical chain to notebook 02's clean_structures() -- copied, not
    reimplemented, so an external compound is cleaned exactly like a
    training compound was.'''
    try:
        mol = Chem.MolFromSmiles(smiles)
        if mol is None:
            return None
        mol = rdMolStandardize.Cleanup(mol)
        mol = rdMolStandardize.FragmentParent(mol)
        mol = rdMolStandardize.Normalizer().normalize(mol)
        mol = rdMolStandardize.Uncharger().uncharge(mol)
        canon_mol = rdMolStandardize.TautomerEnumerator().Canonicalize(mol)
        return Chem.MolToSmiles(canon_mol, canonical=True, isomericSmiles=True)
    except Exception:
        return None


bindingdb_harmonised = {}
for target in TARGETS:
    df_bdb = bindingdb_raw[target]
    if len(df_bdb) == 0:
        print(f'{target}: no BindingDB data, skipping harmonisation')
        bindingdb_harmonised[target] = pd.DataFrame()
        continue

    df_bdb = df_bdb.copy()
    df_bdb['clean_smiles'] = df_bdb['SMILES'].apply(standardize_smiles)
    n_failed = df_bdb['clean_smiles'].isna().sum()
    df_bdb = df_bdb.dropna(subset=['clean_smiles'])

    # -log10(M): affinity_nm is in nM -> M is affinity_nm * 1e-9
    df_bdb['pActivity_single'] = -np.log10(df_bdb['affinity_nm'] * 1e-9)

    # Pool to one potency value per compound -- median across all measurements
    # (any affinity type, any number of replicates), same aggregation logic
    # as notebook 02's own multi-measurement pooling.
    pooled = df_bdb.groupby('clean_smiles').agg(
        pActivity=('pActivity_single', 'median'),
        n_measurements=('pActivity_single', 'count'),
    ).reset_index()
    pooled['activity'] = (pooled['pActivity'] >= ACTIVITY_THRESHOLD).astype(int)

    print(f'{target}: {len(df_bdb)} raw usable records ({n_failed} failed standardisation) '
          f'-> {len(pooled)} unique compounds, {pooled["activity"].sum()} active '
          f'({pooled["activity"].mean()*100:.1f}%)')
    bindingdb_harmonised[target] = pooled

harmonised_path = ext_path / 'bindingdb_harmonised.csv'
_harmonised_frames = [df.assign(target=t) for t, df in bindingdb_harmonised.items() if len(df)]
bindingdb_harmonised_all = (pd.concat(_harmonised_frames, ignore_index=True)
                            if _harmonised_frames else pd.DataFrame(
                                columns=['clean_smiles', 'pActivity', 'n_measurements', 'activity', 'target']))
bindingdb_harmonised_all.to_csv(harmonised_path, index=False)
print(f'\nSaved: {harmonised_path}')


drd2: 569 raw usable records (0 failed standardisation) -> 563 unique compounds, 445 active (79.0%)
cb2: 506 raw usable records (1 failed standardisation) -> 502 unique compounds, 409 active (81.5%)
adora2a: 524 raw usable records (0 failed standardisation) -> 524 unique compounds, 429 active (81.9%)
oprm1: 489 raw usable records (3 failed standardisation) -> 483 unique compounds, 388 active (80.3%)
ccr5: 129 raw usable records (0 failed standardisation) -> 128 unique compounds, 114 active (89.1%)

Saved: /content/drive/My Drive/gpcr_benchmark/ml/results/external_validation/bindingdb_harmonised.csv


## Module D: Training-Set Overlap Check (BindingDB axis)

Same logic as an earlier implementation's Step 3, generalised to 5 targets: exact
`clean_smiles` match against each target's own training data. Overlapping
compounds are **not evaluated on** for the headline BindingDB metric (already
seen during training, scoring on them would inflate performance) but are
kept in the output, flagged, for full transparency -- matches this
project's disclose-don't-drop convention throughout.

In [22]:
bindingdb_overlap_records = []
bindingdb_eval = {}
for target in TARGETS:
    df_bdb = bindingdb_harmonised[target]
    if len(df_bdb) == 0:
        bindingdb_eval[target] = pd.DataFrame()
        continue

    train_path = data_path / 'processed' / f'cleaned_data_{target}_full.csv'
    train_smiles = set(pd.read_csv(train_path)['clean_smiles'])

    df_bdb = df_bdb.copy()
    df_bdb['in_training_set'] = df_bdb['clean_smiles'].isin(train_smiles)
    n_overlap = int(df_bdb['in_training_set'].sum())
    bindingdb_overlap_records.append({
        'target': target, 'n_bindingdb_compounds': len(df_bdb),
        'n_overlap_with_training': n_overlap,
        'pct_overlap': round(n_overlap / len(df_bdb) * 100, 1) if len(df_bdb) else 0.0,
        'n_available_for_evaluation': len(df_bdb) - n_overlap,
    })
    bindingdb_eval[target] = df_bdb[~df_bdb['in_training_set']].reset_index(drop=True)
    print(f'{target}: {n_overlap} / {len(df_bdb)} BindingDB compounds already in training '
          f'({n_overlap / len(df_bdb) * 100:.1f}%) -- {len(bindingdb_eval[target])} available '
          f'for genuinely independent evaluation')

bindingdb_overlap_df = pd.DataFrame(bindingdb_overlap_records)
overlap_path = ext_path / 'bindingdb_training_overlap.csv'
bindingdb_overlap_df.to_csv(overlap_path, index=False)
print(f'\nSaved: {overlap_path}')
print(bindingdb_overlap_df.to_string(index=False))

drd2: 544 / 563 BindingDB compounds already in training (96.6%) -- 19 available for genuinely independent evaluation
cb2: 487 / 502 BindingDB compounds already in training (97.0%) -- 15 available for genuinely independent evaluation
adora2a: 390 / 524 BindingDB compounds already in training (74.4%) -- 134 available for genuinely independent evaluation
oprm1: 434 / 483 BindingDB compounds already in training (89.9%) -- 49 available for genuinely independent evaluation
ccr5: 122 / 128 BindingDB compounds already in training (95.3%) -- 6 available for genuinely independent evaluation

Saved: /content/drive/My Drive/gpcr_benchmark/ml/results/external_validation/bindingdb_training_overlap.csv
 target  n_bindingdb_compounds  n_overlap_with_training  pct_overlap  n_available_for_evaluation
   drd2                    563                      544         96.6                          19
    cb2                    502                      487         97.0                          15
adora2a     

## Module E: Apply Deployed Notebook 3 Models (BindingDB axis)

Loads the real, already-trained artifacts notebook 3 produced -- the
deployed classifier, its Venn-ABERS calibrator, its crepes conformal
wrapper, and its AD reference -- and scores the BindingDB evaluation set
with them. **Not** an earlier implementation's ensemble+different-calibration
approach; this is the exact same pipeline notebook 4 screens DrugBank
with, just pointed at BindingDB compounds instead.

**Algorithm choice is read from `ml/results/best_algorithm_by_combination.csv`
-- the exact same source notebook 4 reads, with the exact same filter
(`target`, `activity_pool`, `task='classification'`) -- not derived
heuristically here.** Two wrong approaches were tried and rejected before
landing on this, both live-verified wrong against real data:

1. **Argmax of `final_model_test_performance.csv`'s test_roc_auc.** Rejected
   because notebook 3's own code explicitly does NOT select the deployed
   algorithm this way -- its `best_algorithm` cell's own comment: "fixes the
   selection-on-test leak." Picking whichever algorithm scored highest on
   the outer test set would itself be a leak; `final_model_test_
   performance.csv` is a separate, later, reporting-only comparison table.
2. **Scanning `ml/models/` for which `*_clf.joblib`/calibrator files exist,
   preferring the test-CSV's pick among what's present.** Rejected because
   it was found live, after a full cluster resync, that notebook 3 actually
   persists artifacts for **all 3 algorithms** for every (target, pool,
   representation) -- needed for its own benchmark comparison -- so file
   existence carries no information about which one was selected as *the*
   deployed model. An earlier version of this notebook built on this
   premise silently used the wrong algorithm for 3 of 5 targets (ADORA2A,
   OPRM1, and effectively also CB2/CCR5 depending on which heuristic ran)
   before this was caught.

`best_algorithm_by_combination.csv` is notebook 3's own persisted record of
its leak-free, inner-CV-based selection -- reading it directly, the same
way notebook 4 does, is the only way this notebook's headline external-
validation numbers are guaranteed to describe the same model notebook 4
actually screens DrugBank with.

In [23]:
def compute_features(smiles_list):
    '''Morgan (r=2, 2048 bits) + 10 physicochemical descriptors + combined --
    identical constants to notebook 3 (MORGAN_RADIUS/MORGAN_NBITS/
    DESCRIPTOR_COLS above), so the feature vector matches what the deployed
    model actually expects.'''
    n = len(smiles_list)
    morgan = np.zeros((n, MORGAN_NBITS), dtype=np.float32)
    desc = np.zeros((n, len(DESCRIPTOR_COLS)), dtype=np.float32)
    parsed_ok = np.zeros(n, dtype=bool)

    for i, smi in enumerate(smiles_list):
        mol = Chem.MolFromSmiles(smi)
        if mol is None:
            continue
        fp = AllChem.GetMorganFingerprintAsBitVect(mol, radius=MORGAN_RADIUS, nBits=MORGAN_NBITS)
        arr = np.zeros((MORGAN_NBITS,), dtype=np.float32)
        for bit in fp.GetOnBits():
            arr[bit] = 1.0
        morgan[i] = arr
        desc[i] = [
            Descriptors.MolWt(mol), Crippen.MolLogP(mol), Descriptors.TPSA(mol),
            Lipinski.NumHDonors(mol), Lipinski.NumHAcceptors(mol), Descriptors.NumRotatableBonds(mol),
            Descriptors.HeavyAtomCount(mol), Descriptors.RingCount(mol),
            Descriptors.NumAromaticRings(mol), Descriptors.FractionCSP3(mol),
        ]
        parsed_ok[i] = True

    desc = np.nan_to_num(desc, nan=0.0, posinf=0.0, neginf=0.0)
    combined = np.hstack([desc, morgan])
    return combined, parsed_ok


ALGO_LABEL_TO_KEY = {'Random Forest': 'rf', 'XGBoost': 'xgb', 'LightGBM': 'lgb'}
ALGO_KEY_TO_LABEL = {v: k for k, v in ALGO_LABEL_TO_KEY.items()}

# Read exactly as notebook 4 does: notebook 3's own leak-free, inner-CV
# selection record. One row per (target, activity_pool, task); task is
# 'classification' or 'regression' -- this notebook only scores classifiers,
# so every lookup filters to task='classification'.
best_algo_path = results_path / 'best_algorithm_by_combination.csv'
assert best_algo_path.exists(), f'notebook 3 output not found: {best_algo_path} -- run notebook 3 first.'
best_algorithm_df = pd.read_csv(best_algo_path)
print(f'Loaded best-algorithm determinations for {len(best_algorithm_df)} (target, pool, task) combinations.')



# Integrity gate: external validation must use the same leak-free deployed
# classifiers selected by notebook 3, not a stale table or test-set argmax.
def assert_deployed_selection_integrity(scope_targets=TARGETS, pool=POOL, representation=REPRESENTATION):
    label_to_stem = {'Random Forest': 'rf', 'XGBoost': 'xgb', 'LightGBM': 'lgb'}
    stem_to_label = {v: k for k, v in label_to_stem.items()}
    failures = []
    for target in scope_targets:
        row = best_algorithm_df[
            (best_algorithm_df['target'] == target) &
            (best_algorithm_df['activity_pool'] == pool) &
            (best_algorithm_df['task'] == 'classification')
        ]
        if len(row) != 1:
            failures.append(f'{target}/{pool}/classification: expected one selection row, found {len(row)}')
            continue
        recorded = str(row.iloc[0]['best_algorithm'])
        history_path = results_path / f'{target}_{pool}' / 'optuna_tuning_history.json'
        if history_path.exists():
            history = json.load(open(history_path))
            values = {}
            for key, entry in history.items():
                if key.endswith('classification') and isinstance(entry, dict) and 'best_value' in entry:
                    stem = key.split('_')[0]
                    if stem in stem_to_label:
                        values[stem_to_label[stem]] = float(entry['best_value'])
            if values:
                recomputed = max(values, key=values.get)
                if recorded != recomputed:
                    failures.append(f'{target}/{pool}/classification: selection table says {recorded}, tuning history says {recomputed}')
        stem = label_to_stem.get(recorded)
        if stem is None:
            failures.append(f'{target}/{pool}/classification: unknown algorithm label {recorded}')
            continue
        model_file = models_path / f'{target}_{pool}' / representation / f'{stem}_clf.joblib'
        if not model_file.exists():
            failures.append(f'{target}/{pool}/classification: deployed artifact missing: {model_file}')
    if failures:
        raise RuntimeError('Deployed-model selection integrity check failed:\n  - ' + '\n  - '.join(failures))
    print(f'Deployed-model selection integrity check passed for {len(scope_targets)} targets.')

assert_deployed_selection_integrity()

def choose_deployed_algorithm(target):
    '''Deployed algorithm for (target, POOL, classification), read directly
    from best_algorithm_by_combination.csv -- notebook 3's own leak-free,
    inner-CV-selected choice, the exact same source and filter notebook 4
    uses (see 04_drugbank_screening.ipynb's get_best_algo_key). Not derived
    from test-set performance, not derived from scanning which files exist
    on disk -- both were tried, both were live-verified wrong (see Module E
    markdown above). Confirms a complete bundle is actually present before
    returning, since that's still a real possible failure mode (partial
    sync, interrupted deployment) -- but existence is a sanity check here,
    never the selection mechanism.'''
    row = best_algorithm_df[
        (best_algorithm_df['target'] == target) & (best_algorithm_df['activity_pool'] == POOL) &
        (best_algorithm_df['task'] == 'classification')
    ]
    if len(row) == 0:
        print(f'  ⚠️  {target}: no best_algorithm_by_combination.csv row for '
              f'(target={target}, activity_pool={POOL}, task=classification)')
        return None

    algo_label = row.iloc[0]['best_algorithm']
    algo_key = ALGO_LABEL_TO_KEY[algo_label]
    rep_dir = models_path / f'{target}_{POOL}' / REPRESENTATION
    required = [rep_dir / f'{algo_key}_clf.joblib', rep_dir / f'{algo_key}_clf_calibrator.pkl',
                rep_dir / 'ad_reference.pkl']
    if not all(p.exists() for p in required):
        print(f'  ⚠️  {target}: deployed algorithm is {algo_label} per best_algorithm_by_combination.csv, '
              f'but its required files are not all present in {rep_dir} -- skipping this target only.')
        return None

    return algo_label


def apply_probability_calibrator(calibrator, raw_proba):
    """Return calibrated active probability plus Venn-Abers interval columns.
    Current notebook-3 reruns use Venn-Abers, but older local artifacts can
    still be Platt LogisticRegression calibrators. Supporting both keeps this
    validation notebook executable while recording which calibration family was
    actually used. Platt has no per-compound validity interval, so p0/p1 are NaN."""
    cal_type = type(calibrator).__name__
    cal_module = type(calibrator).__module__
    if 'venn' in cal_module.lower() or cal_type.lower().startswith('venn'):
        p_prime, p0_p1 = calibrator.predict_proba(raw_proba)
        p_prime = np.asarray(p_prime)
        p0_p1 = np.asarray(p0_p1)
        calibrated = p_prime[:, 1] if p_prime.ndim == 2 else p_prime
        p0 = p0_p1[:, 0] if p0_p1.ndim == 2 else np.full(len(calibrated), np.nan)
        p1 = p0_p1[:, 1] if p0_p1.ndim == 2 and p0_p1.shape[1] > 1 else np.full(len(calibrated), np.nan)
        return calibrated, p0, p1, 'venn_abers'

    # Legacy Platt fallback from earlier notebook-3 artifacts: LogisticRegression
    # was fit on logit(raw positive-class probability), not on the 2-column
    # probability matrix used by Venn-Abers.
    p_raw = np.clip(raw_proba[:, 1], 1e-6, 1 - 1e-6)
    logit_p = np.log(p_raw / (1 - p_raw)).reshape(-1, 1)
    calibrated = calibrator.predict_proba(logit_p)[:, 1]
    return calibrated, np.full(len(calibrated), np.nan), np.full(len(calibrated), np.nan), 'platt_logistic_legacy'

bindingdb_scored = {}
for target in TARGETS:
    df_eval = bindingdb_eval[target]
    if len(df_eval) == 0:
        bindingdb_scored[target] = pd.DataFrame()
        continue

    algo_label = choose_deployed_algorithm(target)
    if algo_label is None:
        print(f'{target}: no usable deployed algorithm, skipping')
        bindingdb_scored[target] = pd.DataFrame()
        continue
    algo = ALGO_LABEL_TO_KEY[algo_label]

    rep_dir = models_path / f'{target}_{POOL}' / REPRESENTATION
    clf_path = rep_dir / f'{algo}_clf.joblib'
    cal_path = rep_dir / f'{algo}_clf_calibrator.pkl'
    conformal_path = rep_dir / f'{algo}_clf_conformal.pkl'
    ad_path = rep_dir / 'ad_reference.pkl'

    X, parsed_ok = compute_features(df_eval['clean_smiles'].tolist())
    df_eval = df_eval[parsed_ok].reset_index(drop=True)
    X = X[parsed_ok]

    clf = joblib.load(clf_path)
    calibrator = joblib.load(cal_path)  # VennAbers object
    ad_ref = joblib.load(ad_path)

    raw_proba = clf.predict_proba(X)
    calibrated_proba, p0, p1, calibration_method = apply_probability_calibrator(calibrator, raw_proba)
    interval_width = p1 - p0

    X_scaled = ad_ref['scaler'].transform(X)
    dists, _ = ad_ref['knn'].kneighbors(X_scaled, n_neighbors=ad_ref['k'])
    knn_mean_dist = dists.mean(axis=1)
    within_ad = knn_mean_dist <= ad_ref['threshold']

    df_eval['algorithm_used'] = algo_label
    df_eval['predicted_proba_raw'] = raw_proba[:, 1]
    df_eval['predicted_proba_calibrated'] = calibrated_proba
    df_eval['calibration_interval_lo'] = p0
    df_eval['calibration_interval_hi'] = p1
    df_eval['calibration_interval_width'] = interval_width
    df_eval['calibration_method'] = calibration_method
    df_eval['within_applicability_domain'] = within_ad
    df_eval['ad_mean_distance'] = knn_mean_dist

    if conformal_path.exists():
        wrapped_clf = joblib.load(conformal_path)
        pred_set = wrapped_clf.predict_set(X, confidence=0.9, labels=False)
        classes = list(clf.classes_)
        active_col = classes.index(1) if 1 in classes else 1
        inactive_col = classes.index(0) if 0 in classes else 0
        df_eval['conformal_includes_inactive_90pct'] = pred_set[:, inactive_col].astype(bool)
        df_eval['conformal_includes_active_90pct'] = pred_set[:, active_col].astype(bool)
        df_eval['conformal_set_size_90pct'] = pred_set.sum(axis=1)

    bindingdb_scored[target] = df_eval
    n_active_pred = int((calibrated_proba >= 0.5).sum())
    n_within_ad = int(within_ad.sum())
    print(f'{target}: {len(df_eval)} scored (algorithm: {algo_label}) -- '
          f'{n_active_pred} predicted active, {n_within_ad} within AD ({n_within_ad/len(df_eval)*100:.0f}%)')

scored_path = ext_path / 'bindingdb_scored.csv'
_scored_frames = [df.assign(target=t) for t, df in bindingdb_scored.items() if len(df)]
bindingdb_scored_all = (pd.concat(_scored_frames, ignore_index=True)
                        if _scored_frames else pd.DataFrame())
bindingdb_scored_all.to_csv(scored_path, index=False)
print(f'\nSaved: {scored_path}')


Loaded best-algorithm determinations for 30 (target, pool, task) combinations.
drd2: 18 scored (algorithm: XGBoost) -- 13 predicted active, 15 within AD (83%)
cb2: 15 scored (algorithm: LightGBM) -- 9 predicted active, 12 within AD (80%)
adora2a: 134 scored (algorithm: Random Forest) -- 115 predicted active, 82 within AD (61%)
oprm1: 49 scored (algorithm: Random Forest) -- 35 predicted active, 40 within AD (82%)
ccr5: 6 scored (algorithm: Random Forest) -- 1 predicted active, 6 within AD (100%)

Saved: /content/drive/My Drive/gpcr_benchmark/ml/results/external_validation/bindingdb_scored.csv


## Module F: Temporal Holdout (3-layer stratification)

Per-target cutoff year chosen from that target's own `document_year`
distribution (~20% of compounds after cutoff, not a single global year
forced onto all 5). Uses the **already-deployed**, already-scaffold-split
model -- no retraining. Three explicit strata, not one:

1. `post_cutoff_all` -- every compound reported after the cutoff, regardless
   of scaffold-split assignment.
2. `post_cutoff_never_seen` -- post-cutoff compounds whose `global_compound_id`
   landed in the scaffold-split **test** fold (notebook 3's
   `train_test_split.csv`) -- genuinely never touched during training or
   calibration.
3. `post_cutoff_seen_in_training` -- post-cutoff compounds that landed in
   `cal_train`/`cal_holdout` -- these WERE used to fit or calibrate the
   deployed model even though they were published after the nominal cutoff
   (a real, expected consequence of scaffold splitting being blind to
   publication date). Reported separately, not dropped -- contamination
   made visible, not assumed away, matching this project's disclose-don't-
   drop convention (see Module D).

Stratum 2 is the headline "would this model have worked on compounds that
didn't exist yet" number; stratum 3 is reported explicitly as a positive-
control sanity check (should score noticeably better than stratum 2 --
if it doesn't, that is itself a finding).

In [24]:
from sklearn.metrics import roc_auc_score

TARGET_CHEMBL_ID = {
    'drd2': 'CHEMBL217', 'cb2': 'CHEMBL253', 'adora2a': 'CHEMBL251',
    'oprm1': 'CHEMBL233', 'ccr5': 'CHEMBL274',
}
CHEMBL_STANDARD_TYPES = ['Ki', 'IC50', 'EC50']  # matches notebook 01's own pull


def fetch_document_years(target_chembl_id: str) -> pd.DataFrame:
    """Paginated ChEMBL activity fetch, fields limited to molecule_chembl_id +
    document_year -- same target/standard_type filter as notebook 01's original
    pull, just requesting the one extra field that pull never asked for."""
    records = []
    base = 'https://www.ebi.ac.uk/chembl/api/data/activity.json'
    for std_type in CHEMBL_STANDARD_TYPES:
        offset = 0
        limit = 1000
        while True:
            url = (f'{base}?target_chembl_id={target_chembl_id}&standard_type={std_type}'
                   f'&limit={limit}&offset={offset}'
                   f'&only=molecule_chembl_id,document_year')
            try:
                with urllib.request.urlopen(url, timeout=60) as resp:
                    payload = json.load(resp)
            except (urllib.error.URLError, urllib.error.HTTPError) as exc:
                print(f'  ⚠️  {target_chembl_id}/{std_type} offset={offset}: fetch failed -- {exc}')
                break
            activities = payload.get('activities', [])
            for a in activities:
                if a.get('document_year') is not None and a.get('molecule_chembl_id'):
                    records.append({'molecule_chembl_id': a['molecule_chembl_id'],
                                     'document_year': int(a['document_year'])})
            total_count = payload.get('page_meta', {}).get('total_count', 0)
            offset += limit
            if offset >= total_count or len(activities) == 0:
                break
    return pd.DataFrame(records)


global_id_map = pd.read_csv(data_path / 'processed' / 'chemical_space_pca_coordinates.csv',
                             usecols=['clean_smiles', 'target', 'global_compound_id'])
global_id_map = global_id_map.drop_duplicates(subset=['target', 'clean_smiles', 'global_compound_id'])


def load_split_assignments(target: str, df_with_ids: pd.DataFrame) -> pd.DataFrame:
    """Return global_compound_id + split for the target full-pool deployment.
    Prefer train_test_split.csv. If absent, reconstruct from notebook-3 index
    arrays against cleaned-data row order after global_id merge. Raises
    FileNotFoundError if neither source is available -- caller is responsible
    for catching this per-target so one target's missing artifacts can't abort
    the other 4 (same per-unit-isolation convention as notebook 05's docking
    loop)."""
    split_path = results_path / f'{target}_full' / 'train_test_split.csv'
    if split_path.exists():
        split_df = pd.read_csv(split_path)
        split_df['split_source'] = 'train_test_split.csv'
        return split_df

    shard_dir = results_path / f'{target}_full'
    cal_train_path = shard_dir / 'cal_train_indices.npy'
    cal_holdout_path = shard_dir / 'cal_holdout_indices.npy'
    test_path = shard_dir / 'test_indices.npy'
    if not all(p.exists() for p in [cal_train_path, cal_holdout_path, test_path]):
        raise FileNotFoundError(f'No train_test_split.csv or complete index-array split files for {target}')

    split_recon = df_with_ids[['global_compound_id']].copy().reset_index(drop=True)
    split_recon['split'] = 'unassigned'
    split_recon.loc[np.load(cal_train_path), 'split'] = 'cal_train'
    split_recon.loc[np.load(cal_holdout_path), 'split'] = 'cal_holdout'
    split_recon.loc[np.load(test_path), 'split'] = 'test'
    split_recon = split_recon[split_recon['split'] != 'unassigned'].dropna(subset=['global_compound_id'])
    split_recon = split_recon.drop_duplicates(subset=['global_compound_id'], keep='first')
    split_recon['split_source'] = 'reconstructed_from_index_arrays'
    print(f'  ⚠️  {target}: train_test_split.csv missing; reconstructed split labels from index arrays')
    return split_recon


temporal_scored = {}
temporal_cutoffs = {}
temporal_split_source_records = []
for target in TARGETS:
    year_cache_path = ext_data_path.parent / 'chembl_document_years' / f'document_years_{target}.csv'
    year_cache_path.parent.mkdir(parents=True, exist_ok=True)
    if year_cache_path.exists():
        years_df = pd.read_csv(year_cache_path)
    else:
        print(f'Fetching document_year for {target.upper()} ({TARGET_CHEMBL_ID[target]})...')
        years_df = fetch_document_years(TARGET_CHEMBL_ID[target])
        years_df.to_csv(year_cache_path, index=False)
    years_df = years_df.groupby('molecule_chembl_id', as_index=False)['document_year'].min()

    df_clean = pd.read_csv(data_path / 'processed' / f'cleaned_data_{target}_full.csv')
    df_clean = df_clean.merge(years_df, on='molecule_chembl_id', how='left')
    n_missing_year = df_clean['document_year'].isna().sum()

    id_map_t = global_id_map[global_id_map['target'] == target][['clean_smiles', 'global_compound_id']]
    id_map_t = id_map_t.drop_duplicates(subset=['clean_smiles'], keep='first')
    df_clean = df_clean.merge(id_map_t, on='clean_smiles', how='left')

    try:
        split_df = load_split_assignments(target, df_clean)
    except FileNotFoundError as exc:
        print(f'  ⚠️  {target}: {exc} -- skipping temporal axis for this target only '
              f'(BindingDB axis and other targets are unaffected).')
        temporal_split_source_records.append({
            'target': target, 'split_source': 'unavailable', 'n_split_rows': 0,
        })
        temporal_scored[target] = pd.DataFrame()
        continue

    temporal_split_source_records.append({
        'target': target,
        'split_source': split_df['split_source'].iloc[0] if 'split_source' in split_df.columns and len(split_df) else 'unknown',
        'n_split_rows': len(split_df),
    })
    split_df = split_df[['global_compound_id', 'split']].drop_duplicates(subset=['global_compound_id'])
    df_clean = df_clean.merge(split_df, on='global_compound_id', how='left')

    df_dated = df_clean.dropna(subset=['document_year', 'split']).copy()
    df_dated['document_year'] = df_dated['document_year'].astype(int)

    if len(df_dated) == 0:
        print(f'{target}: no dated compounds with split assignment; skipping temporal axis')
        temporal_scored[target] = pd.DataFrame()
        continue

    cutoff_year = int(df_dated['document_year'].quantile(1 - TEMPORAL_HOLDOUT_FRACTION_TARGET))
    temporal_cutoffs[target] = cutoff_year
    print(f'{target}: cutoff year = {cutoff_year} ({n_missing_year} compounds missing document_year, excluded)')

    post_cutoff = df_dated[df_dated['document_year'] > cutoff_year].reset_index(drop=True)
    post_cutoff['stratum'] = np.where(post_cutoff['split'] == 'test',
                                       'post_cutoff_never_seen', 'post_cutoff_seen_in_training')

    algo_label = choose_deployed_algorithm(target)
    if algo_label is None or len(post_cutoff) == 0:
        print(f'{target}: skipping scoring (no algorithm or empty post-cutoff set)')
        temporal_scored[target] = pd.DataFrame()
        continue
    algo = ALGO_LABEL_TO_KEY[algo_label]
    rep_dir = models_path / f'{target}_{POOL}' / REPRESENTATION
    clf_path, cal_path, conformal_path, ad_path = (
        rep_dir / f'{algo}_clf.joblib', rep_dir / f'{algo}_clf_calibrator.pkl',
        rep_dir / f'{algo}_clf_conformal.pkl', rep_dir / 'ad_reference.pkl')
    if not all(p.exists() for p in (clf_path, cal_path, ad_path)):
        print(f'{target}: missing deployed artifacts, skipping')
        temporal_scored[target] = pd.DataFrame()
        continue

    X, parsed_ok = compute_features(post_cutoff['clean_smiles'].tolist())
    post_cutoff = post_cutoff[parsed_ok].reset_index(drop=True)
    X = X[parsed_ok]

    clf = joblib.load(clf_path)
    calibrator = joblib.load(cal_path)
    ad_ref = joblib.load(ad_path)

    raw_proba = clf.predict_proba(X)
    calibrated_proba, p0, p1, calibration_method = apply_probability_calibrator(calibrator, raw_proba)
    post_cutoff['algorithm_used'] = algo_label
    post_cutoff['predicted_proba_raw'] = raw_proba[:, 1]
    post_cutoff['predicted_proba_calibrated'] = calibrated_proba
    post_cutoff['calibration_interval_lo'] = p0
    post_cutoff['calibration_interval_hi'] = p1
    post_cutoff['calibration_interval_width'] = p1 - p0
    post_cutoff['calibration_method'] = calibration_method

    X_scaled = ad_ref['scaler'].transform(X)
    dists, _ = ad_ref['knn'].kneighbors(X_scaled, n_neighbors=ad_ref['k'])
    post_cutoff['ad_mean_distance'] = dists.mean(axis=1)
    post_cutoff['within_applicability_domain'] = post_cutoff['ad_mean_distance'] <= ad_ref['threshold']

    if conformal_path.exists():
        wrapped_clf = joblib.load(conformal_path)
        pred_set = wrapped_clf.predict_set(X, confidence=0.9, labels=False)
        classes = list(clf.classes_)
        active_col = classes.index(1) if 1 in classes else 1
        inactive_col = classes.index(0) if 0 in classes else 0
        post_cutoff['conformal_includes_inactive_90pct'] = pred_set[:, inactive_col].astype(bool)
        post_cutoff['conformal_includes_active_90pct'] = pred_set[:, active_col].astype(bool)
        post_cutoff['conformal_set_size_90pct'] = pred_set.sum(axis=1)

    temporal_scored[target] = post_cutoff
    for stratum in ['post_cutoff_all', 'post_cutoff_never_seen', 'post_cutoff_seen_in_training']:
        sub = post_cutoff if stratum == 'post_cutoff_all' else post_cutoff[post_cutoff['stratum'] == stratum]
        if len(sub) > 1 and sub['activity'].nunique() > 1:
            auc = roc_auc_score(sub['activity'], sub['predicted_proba_calibrated'])
            print(f'  {target}/{stratum}: n={len(sub)}, ROC-AUC={auc:.3f}')
        else:
            print(f'  {target}/{stratum}: n={len(sub)} (insufficient for ROC-AUC)')

cutoffs_df = pd.DataFrame([{'target': t, 'cutoff_year': y} for t, y in temporal_cutoffs.items()])
cutoffs_df.to_csv(ext_path / 'temporal_cutoff_years.csv', index=False)
pd.DataFrame(temporal_split_source_records).to_csv(ext_path / 'temporal_split_sources.csv', index=False)

temporal_scored_path = ext_path / 'temporal_scored.csv'
_temporal_frames = [df.assign(target=t) for t, df in temporal_scored.items() if len(df)]
temporal_scored_all = (pd.concat(_temporal_frames, ignore_index=True) if _temporal_frames else pd.DataFrame())
temporal_scored_all.to_csv(temporal_scored_path, index=False)
print(f'\nSaved: {ext_path / "temporal_cutoff_years.csv"}')
print(f'Saved: {ext_path / "temporal_split_sources.csv"}')
print(f'Saved: {temporal_scored_path}')


drd2: cutoff year = 2019 (38 compounds missing document_year, excluded)
  drd2/post_cutoff_all: n=2169, ROC-AUC=0.966
  drd2/post_cutoff_never_seen: n=425, ROC-AUC=0.863
  drd2/post_cutoff_seen_in_training: n=1744, ROC-AUC=0.982
cb2: cutoff year = 2017 (159 compounds missing document_year, excluded)
  cb2/post_cutoff_all: n=1800, ROC-AUC=0.968
  cb2/post_cutoff_never_seen: n=337, ROC-AUC=0.928
  cb2/post_cutoff_seen_in_training: n=1463, ROC-AUC=0.974
adora2a: cutoff year = 2021 (20 compounds missing document_year, excluded)
  adora2a/post_cutoff_all: n=1562, ROC-AUC=0.976
  adora2a/post_cutoff_never_seen: n=398, ROC-AUC=0.919
  adora2a/post_cutoff_seen_in_training: n=1164, ROC-AUC=0.991
oprm1: cutoff year = 2020 (800 compounds missing document_year, excluded)
  oprm1/post_cutoff_all: n=1258, ROC-AUC=0.983
  oprm1/post_cutoff_never_seen: n=230, ROC-AUC=0.888
  oprm1/post_cutoff_seen_in_training: n=1028, ROC-AUC=0.992
ccr5: cutoff year = 2013 (2 compounds missing document_year, excluded)

## Module G: Final Combined Metrics (BindingDB + Temporal)

One tidy long-format table, both axes side by side, per target -- the
actual publication-ready output of this notebook. Metrics: ROC-AUC, PR-AUC,
Brier score (against calibrated probability), 90%-confidence conformal
empirical coverage, and % within applicability domain. Computed only where
both classes are present in a stratum (small strata can't support ROC-AUC);
flagged `n_too_small` rather than silently omitted.

In [25]:
from sklearn.metrics import roc_auc_score, average_precision_score, brier_score_loss

N_BOOTSTRAP_EXTERNAL_VALIDATION = 1000
MIN_BINDINGDB_N_FOR_PRIMARY_CLAIM = 30
MIN_CLASS_N_FOR_PRIMARY_CLAIM = 5


def _metric_value(y, p, metric):
    if len(y) < 2 or pd.Series(y).nunique() < 2:
        return np.nan
    if metric == 'roc_auc':
        return roc_auc_score(y, p)
    if metric == 'pr_auc':
        return average_precision_score(y, p)
    if metric == 'brier':
        return brier_score_loss(y, p)
    raise ValueError(metric)


def bootstrap_metric_cis(df_stratum, label_col='activity', proba_col='predicted_proba_calibrated',
                         n_bootstrap=N_BOOTSTRAP_EXTERNAL_VALIDATION, random_state=RANDOM_STATE):
    """Nonparametric bootstrap CIs for finite external/temporal strata.
    Resamples rows with replacement and skips bootstrap draws that contain only
    one class for ROC-AUC/PR-AUC. This is especially important for BindingDB,
    where several targets have very small de-overlapped n."""
    out = {f'{m}_ci_low': np.nan for m in ['roc_auc', 'pr_auc', 'brier']}
    out.update({f'{m}_ci_high': np.nan for m in ['roc_auc', 'pr_auc', 'brier']})
    out['n_bootstrap_successful'] = 0

    n = len(df_stratum)
    if n < 8 or df_stratum[label_col].nunique() < 2:
        return out

    rng = np.random.default_rng(random_state)
    y_all = df_stratum[label_col].to_numpy()
    p_all = df_stratum[proba_col].to_numpy()
    values = {'roc_auc': [], 'pr_auc': [], 'brier': []}
    for _ in range(n_bootstrap):
        idx = rng.integers(0, n, size=n)
        y = y_all[idx]
        p = p_all[idx]
        if len(np.unique(y)) < 2:
            continue
        for metric in values:
            values[metric].append(_metric_value(y, p, metric))

    successful = len(values['roc_auc'])
    out['n_bootstrap_successful'] = successful
    if successful == 0:
        return out
    for metric, vals in values.items():
        vals = np.asarray(vals, dtype=float)
        out[f'{metric}_ci_low'] = float(np.nanpercentile(vals, 2.5))
        out[f'{metric}_ci_high'] = float(np.nanpercentile(vals, 97.5))
    return out


def interpretation_flag(axis, stratum, n, n_active, n_inactive, base_flag=''):
    if base_flag:
        return base_flag
    if n < 2 or min(n_active, n_inactive) == 0:
        return 'n_too_small'
    if axis == 'bindingdb':
        if n < MIN_BINDINGDB_N_FOR_PRIMARY_CLAIM or min(n_active, n_inactive) < MIN_CLASS_N_FOR_PRIMARY_CLAIM:
            return 'small_n_descriptive'
        return 'external_database_deoverlapped'
    if axis == 'temporal':
        if stratum == 'post_cutoff_never_seen':
            return 'temporal_never_seen_primary'
        if stratum == 'post_cutoff_seen_in_training':
            return 'contaminated_reference_only'
        if stratum == 'post_cutoff_all':
            return 'mixed_temporal_context'
    return ''


def compute_stratum_metrics(df_stratum, axis='', stratum='', label_col='activity',
                            proba_col='predicted_proba_calibrated'):
    n = len(df_stratum)
    n_active = int(df_stratum[label_col].sum()) if n and label_col in df_stratum.columns else 0
    n_inactive = int(n - n_active)
    if n < 2 or df_stratum[label_col].nunique() < 2:
        return {
            'n': n, 'n_active': n_active, 'n_inactive': n_inactive,
            'active_prevalence': float(n_active / n) if n else np.nan,
            'roc_auc': np.nan, 'pr_auc': np.nan, 'brier': np.nan,
            'pct_within_ad': np.nan,
            'conformal_coverage_90pct': np.nan,
            'conformal_empirical_coverage_90pct': np.nan,
            'conformal_balanced_coverage_90pct': np.nan,
            'flag': interpretation_flag(axis, stratum, n, n_active, n_inactive, 'n_too_small'),
            **bootstrap_metric_cis(df_stratum, label_col, proba_col),
        }

    y = df_stratum[label_col]
    p = df_stratum[proba_col]
    metrics = {
        'n': n,
        'n_active': n_active,
        'n_inactive': n_inactive,
        'active_prevalence': float(n_active / n),
        'roc_auc': roc_auc_score(y, p),
        'pr_auc': average_precision_score(y, p),
        'brier': brier_score_loss(y, p),
        'pct_within_ad': round(df_stratum['within_applicability_domain'].mean() * 100, 1),
    }

    if {'conformal_includes_active_90pct', 'conformal_includes_inactive_90pct'}.issubset(df_stratum.columns):
        active_mask = y == 1
        inactive_mask = y == 0
        true_label_included = np.where(
            active_mask,
            df_stratum['conformal_includes_active_90pct'].astype(bool),
            df_stratum['conformal_includes_inactive_90pct'].astype(bool),
        )
        cov_active = df_stratum.loc[active_mask, 'conformal_includes_active_90pct'].mean() if active_mask.any() else np.nan
        cov_inactive = df_stratum.loc[inactive_mask, 'conformal_includes_inactive_90pct'].mean() if inactive_mask.any() else np.nan
        empirical_cov = float(np.mean(true_label_included))
        balanced_cov = float(np.nanmean([cov_active, cov_inactive]))
        metrics['conformal_empirical_coverage_90pct'] = empirical_cov
        metrics['conformal_balanced_coverage_90pct'] = balanced_cov
        metrics['conformal_coverage_90pct'] = empirical_cov  # backward-compatible alias, now sample-level empirical coverage
    else:
        metrics['conformal_empirical_coverage_90pct'] = np.nan
        metrics['conformal_balanced_coverage_90pct'] = np.nan
        metrics['conformal_coverage_90pct'] = np.nan

    metrics.update(bootstrap_metric_cis(df_stratum, label_col, proba_col))
    metrics['flag'] = interpretation_flag(axis, stratum, n, n_active, n_inactive)
    return metrics


summary_rows = []

# BindingDB axis
for target in TARGETS:
    df_t = bindingdb_scored.get(target, pd.DataFrame())
    if len(df_t) == 0:
        summary_rows.append({'target': target, 'axis': 'bindingdb', 'stratum': 'all', 'n': 0, 'flag': 'no_data'})
        continue
    m = compute_stratum_metrics(df_t, axis='bindingdb', stratum='all')
    summary_rows.append({'target': target, 'axis': 'bindingdb', 'stratum': 'all', **m})

# Temporal axis, all + clean + contaminated/reference strata
for target in TARGETS:
    df_t = temporal_scored.get(target, pd.DataFrame())
    if len(df_t) == 0:
        summary_rows.append({'target': target, 'axis': 'temporal', 'stratum': 'post_cutoff_all', 'n': 0, 'flag': 'no_data'})
        summary_rows.append({'target': target, 'axis': 'temporal', 'stratum': 'post_cutoff_never_seen', 'n': 0, 'flag': 'no_data'})
        summary_rows.append({'target': target, 'axis': 'temporal', 'stratum': 'post_cutoff_seen_in_training', 'n': 0, 'flag': 'no_data'})
        continue
    for stratum in ['post_cutoff_all', 'post_cutoff_never_seen', 'post_cutoff_seen_in_training']:
        sub = df_t if stratum == 'post_cutoff_all' else df_t[df_t['stratum'] == stratum]
        m = compute_stratum_metrics(sub, axis='temporal', stratum=stratum)
        summary_rows.append({'target': target, 'axis': 'temporal', 'stratum': stratum, **m})

external_validation_summary = pd.DataFrame(summary_rows)
summary_path = ext_path / 'external_validation_summary.csv'
external_validation_summary.to_csv(summary_path, index=False)
print(f'Saved: {summary_path}\n')
_display_cols = [c for c in [
    'target', 'axis', 'stratum', 'n', 'n_active', 'n_inactive', 'roc_auc',
    'roc_auc_ci_low', 'roc_auc_ci_high', 'pr_auc', 'brier', 'pct_within_ad',
    'conformal_empirical_coverage_90pct', 'flag'
] if c in external_validation_summary.columns]
print(external_validation_summary[_display_cols].to_string(index=False))


Saved: /content/drive/My Drive/gpcr_benchmark/ml/results/external_validation/external_validation_summary.csv

 target      axis                      stratum    n  n_active  n_inactive  roc_auc  roc_auc_ci_low  roc_auc_ci_high   pr_auc    brier  pct_within_ad  conformal_empirical_coverage_90pct                           flag
   drd2 bindingdb                          all   18        16           2 0.921875        0.764706         1.000000 0.985243 0.093780           83.3                            1.000000            small_n_descriptive
    cb2 bindingdb                          all   15         9           6 1.000000        1.000000         1.000000 1.000000 0.051678           80.0                            1.000000            small_n_descriptive
adora2a bindingdb                          all  134       109          25 0.850092        0.751386         0.930714 0.949250 0.107949           61.2                            0.917910 external_database_deoverlapped
  oprm1 bindingdb         

## Module H: Manifest


In [26]:
# SHA-256 manifest for external-validation artifacts, matching the locked-output
# convention used throughout the project.
manifest_outputs = {}
for name, path in [
    ('bindingdb_harmonised.csv', harmonised_path),
    ('bindingdb_training_overlap.csv', overlap_path),
    ('bindingdb_scored.csv', scored_path),
    ('temporal_cutoff_years.csv', ext_path / 'temporal_cutoff_years.csv'),
    ('temporal_split_sources.csv', ext_path / 'temporal_split_sources.csv'),
    ('temporal_scored.csv', temporal_scored_path),
    ('external_validation_summary.csv', summary_path),
]:
    path = Path(path)
    if path.exists():
        manifest_outputs[name] = {
            'path': str(path),
            'sha256': _sha256_file(path),
            'n_rows': sum(1 for _ in open(path)) - 1,
        }

manifest = {
    'timestamp': datetime.datetime.now().isoformat(),
    'python_version': platform.python_version(),
    'targets': TARGETS,
    'pool': POOL,
    'representation': REPRESENTATION,
    'bindingdb_cutoff_nm': BINDINGDB_CUTOFF_NM,
    'bindingdb_affinity_types_included': BINDINGDB_AFFINITY_TYPES,
    'bindingdb_affinity_types_excluded': ['Kd'],
    'temporal_framing': 'temporal stress test of deployed scaffold-split models; not a fully temporally retrained model',
    'temporal_holdout_fraction_target': TEMPORAL_HOLDOUT_FRACTION_TARGET,
    'n_bootstrap_external_validation': N_BOOTSTRAP_EXTERNAL_VALIDATION,
    'min_bindingdb_n_for_primary_claim': MIN_BINDINGDB_N_FOR_PRIMARY_CLAIM,
    'min_class_n_for_primary_claim': MIN_CLASS_N_FOR_PRIMARY_CLAIM,
    'calibration_methods_observed': (sorted(bindingdb_scored_all['calibration_method'].dropna().unique().tolist())
                                     if 'bindingdb_scored_all' in globals() and len(bindingdb_scored_all)
                                     and 'calibration_method' in bindingdb_scored_all.columns else []),
    'outputs': manifest_outputs,
}
manifest_path = ext_path / 'manifest_06_external_validation.json'
with open(manifest_path, 'w') as f:
    json.dump(manifest, f, indent=2, default=str)
print(f'Manifest saved: {manifest_path}')


Manifest saved: /content/drive/My Drive/gpcr_benchmark/ml/results/external_validation/manifest_06_external_validation.json


## Summary

Two independent external-validation axes, both applied to the already-
deployed, never-retrained notebook 3 models, across all 5 targets:

- **BindingDB** (different database entirely) -- headline generalisation
  number per target, training-set overlaps disclosed and excluded.
- **Temporal holdout** (post-cutoff ChEMBL records, same database) -- a
  harder, scaffold-split-blind-to-time stress test, split into
  genuinely-never-seen vs. was-in-training-anyway strata rather than
  reported as one contaminated number.

Every intermediate artifact (`bindingdb_harmonised.csv`,
`bindingdb_training_overlap.csv`, `bindingdb_scored.csv`,
`temporal_cutoff_years.csv`, `temporal_scored.csv`,
`external_validation_summary.csv`) is persisted to
`ml/results/external_validation/` -- nothing here needs recomputation to
regenerate the manuscript table.